# 🇳🇵 Nepali Corpus Collection (Model L)

**Phase 1 — Data Collection Pipeline**

This notebook collects ~200M-500M tokens of clean Nepali text:
- **Step 1**: Download automated corpora from HuggingFace (80%)
- **Step 2**: Download + process Wikipedia dump (manual 20%)
- **Step 3**: Deduplicate + merge
- **Step 4**: Create train/val/test splits
- **Step 5**: Train BPE tokenizer
- **Step 6**: Count tokens + upload to Google Drive

**Runtime**: CPU (no GPU needed) | **Time**: ~3 hours

## 0. Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Create output directory on Drive
import os
DRIVE_BASE = '/content/drive/MyDrive/24KaratAu/nepali'
for d in ['raw/wiki_dump', 'raw/hf_automated', 'cleaned/manual', 'cleaned/automated', 'cleaned/merged', 'splits', 'tokenizer']:
    os.makedirs(f'{DRIVE_BASE}/{d}', exist_ok=True)
print('✅ Drive directories created')

In [ ]:
# Install dependencies
!pip install -q wikiextractor datasets sentencepiece
print('✅ Dependencies installed')

In [ ]:
# Common imports and helpers
import json
import hashlib
import re
import random
import bz2
import xml.etree.ElementTree as ET
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

LANGUAGE = 'nepali'
WORK_DIR = '/content/nepali_corpus'
os.makedirs(WORK_DIR, exist_ok=True)

def devanagari_ratio(text):
    if not text: return 0.0
    return sum(1 for c in text if '\u0900' <= c <= '\u097F') / len(text)

def clean_web_text(text):
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'\S+@\S+\.\S+', '', text)
    text = re.sub(r'&[a-zA-Z]+;', ' ', text)
    text = re.sub(r'&#\d+;', ' ', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def text_hash(text):
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

print(f'✅ Working directory: {WORK_DIR}')
print(f'✅ Drive output: {DRIVE_BASE}')

---
## Step 1: Download Automated Corpora (HuggingFace)

We download the automated 80% first so we know what content exists.
Then manual collection (Step 2) avoids duplicating this content.

In [ ]:
from datasets import load_dataset

# ── Configuration ──
AUTO_DIR = f'{WORK_DIR}/automated'
os.makedirs(AUTO_DIR, exist_ok=True)

MIN_CHARS = 100
MIN_DEV_RATIO = 0.5
SHARD_SIZE = 10000
# Target: collect as much as possible. For Nepali, we may not reach 500M tokens.
# Collect everything available.
MAX_CHARS = 5_000_000_000  # 5B chars safety cap

auto_hashes = set()  # Track content hashes for dedup against manual later
auto_total_chars = 0
auto_total_docs = 0
auto_shard_docs = []
auto_shard_idx = 0

def flush_auto_shard():
    global auto_shard_docs, auto_shard_idx
    if not auto_shard_docs:
        return
    shard_file = f'{AUTO_DIR}/auto_shard_{auto_shard_idx:04d}.jsonl'
    with open(shard_file, 'w', encoding='utf-8') as f:
        for doc in auto_shard_docs:
            f.write(json.dumps(doc, ensure_ascii=False) + '\n')
    print(f'  ✅ Wrote {shard_file}: {len(auto_shard_docs):,} docs')
    auto_shard_docs = []
    auto_shard_idx += 1

def process_auto_doc(text, source_name):
    global auto_total_chars, auto_total_docs, auto_shard_docs
    
    cleaned = clean_web_text(text)
    if len(cleaned) < MIN_CHARS:
        return False
    if devanagari_ratio(cleaned) < MIN_DEV_RATIO:
        return False
    
    h = text_hash(cleaned)
    if h in auto_hashes:
        return False
    auto_hashes.add(h)
    
    doc = {
        'doc_id': hashlib.md5(h.encode()).hexdigest()[:16],
        'source': source_name,
        'source_type': f'hf_{source_name}',
        'collection_type': 'automated',
        'language': LANGUAGE,
        'text': cleaned,
        'clean_text_characters': len(cleaned),
        'devanagari_ratio': round(devanagari_ratio(cleaned), 4),
        'text_sha256': h,
    }
    auto_shard_docs.append(doc)
    auto_total_chars += len(cleaned)
    auto_total_docs += 1
    
    if len(auto_shard_docs) >= SHARD_SIZE:
        flush_auto_shard()
    
    if auto_total_docs % 10000 == 0:
        print(f'  [{auto_total_docs:,} docs] {auto_total_chars:,} chars | hashes: {len(auto_hashes):,}')
    
    return True

print('Ready to stream automated corpora')

In [ ]:
# ── 1a: Try Sangraha (ai4bharat) ──
print('=' * 60)
print('Streaming ai4bharat/sangraha for Nepali...')
print('=' * 60)

try:
    ds = load_dataset('ai4bharat/sangraha', 'verified', split='train',
                      streaming=True, trust_remote_code=True)
    for example in ds:
        lang = example.get('language', '') or example.get('lang', '')
        if lang not in ('npi', 'ne', 'nep', 'Nepali', 'nepali'):
            continue
        text = example.get('text', '')
        if text:
            process_auto_doc(text, 'sangraha')
        if auto_total_chars >= MAX_CHARS:
            break
    flush_auto_shard()
    print(f'\nSangraha done: {auto_total_docs:,} docs, {auto_total_chars:,} chars')
except Exception as e:
    print(f'Sangraha error: {e}')
    print('Continuing with other sources...')

In [ ]:
# ── 1b: Try CulturaX ──
print('=' * 60)
print('Streaming uonlp/CulturaX for Nepali...')
print('=' * 60)

before_docs = auto_total_docs
try:
    ds = load_dataset('uonlp/CulturaX', 'ne', split='train',
                      streaming=True, trust_remote_code=True)
    for example in ds:
        text = example.get('text', '')
        if text:
            process_auto_doc(text, 'culturax')
        if auto_total_chars >= MAX_CHARS:
            break
    flush_auto_shard()
    print(f'\nCulturaX added: {auto_total_docs - before_docs:,} new docs')
except Exception as e:
    print(f'CulturaX error: {e}')
    print('Continuing...')

In [ ]:
# ── 1c: Try CC-100 ──
print('=' * 60)
print('Streaming CC-100 for Nepali...')
print('=' * 60)

before_docs = auto_total_docs
try:
    ds = load_dataset('cc100', 'ne', split='train',
                      streaming=True, trust_remote_code=True)
    for example in ds:
        text = example.get('text', '')
        if text:
            process_auto_doc(text, 'cc100')
        if auto_total_chars >= MAX_CHARS:
            break
    flush_auto_shard()
    print(f'\nCC-100 added: {auto_total_docs - before_docs:,} new docs')
except Exception as e:
    print(f'CC-100 error: {e}')
    print('Continuing...')

In [ ]:
# ── Automated collection summary ──
print('\n' + '=' * 60)
print(f'AUTOMATED COLLECTION COMPLETE')
print(f'  Total documents: {auto_total_docs:,}')
print(f'  Total characters: {auto_total_chars:,}')
print(f'  Unique hashes: {len(auto_hashes):,}')
print(f'  Shards written: {auto_shard_idx}')
print(f'  Est. tokens (÷6): ~{auto_total_chars // 6:,}')
print('=' * 60)

# Copy automated shards to Drive (raw backup)
!cp -r {AUTO_DIR}/*.jsonl {DRIVE_BASE}/raw/hf_automated/ 2>/dev/null || true
print('✅ Automated shards backed up to Drive')

---
## Step 2: Wikipedia Dump (Manual Collection)

Now we download and process the Nepali Wikipedia dump.
We skip articles whose content hash already exists in the automated set.

In [ ]:
# Download Nepali Wikipedia dump (~50MB)
WIKI_DUMP = f'{WORK_DIR}/newiki-latest-pages-articles.xml.bz2'
if not os.path.exists(WIKI_DUMP):
    print('Downloading Nepali Wikipedia dump...')
    !wget -q --show-progress -O {WIKI_DUMP} \
        'https://dumps.wikimedia.org/newiki/latest/newiki-latest-pages-articles.xml.bz2'
    size_mb = os.path.getsize(WIKI_DUMP) / (1024*1024)
    print(f'✅ Downloaded: {size_mb:.1f} MB')
else:
    size_mb = os.path.getsize(WIKI_DUMP) / (1024*1024)
    print(f'Already downloaded: {size_mb:.1f} MB')

# Also save raw dump to Drive
!cp {WIKI_DUMP} {DRIVE_BASE}/raw/wiki_dump/ 2>/dev/null || true
print('✅ Raw dump backed up to Drive')

In [ ]:
# ── Wiki markup cleaning ──
WIKI_MARKUP = [
    (re.compile(r'\{\{[^}]*\}\}'), ''),
    (re.compile(r'\[\[Category:[^\]]*\]\]'), ''),
    (re.compile(r'\[\[श्रेणी:[^\]]*\]\]'), ''),
    (re.compile(r'\[\[[^|\]]*\|([^\]]*)\]\]'), r'\1'),
    (re.compile(r'\[\[([^\]]*)\]\]'), r'\1'),
    (re.compile(r'\[https?://[^\s\]]*\s*([^\]]*)\]'), r'\1'),
    (re.compile(r'\[https?://[^\]]*\]'), ''),
    (re.compile(r"'{2,5}"), ''),
    (re.compile(r'<ref[^>]*>.*?</ref>', re.DOTALL), ''),
    (re.compile(r'<ref[^/>]*/>'), ''),
    (re.compile(r'<[^>]+>'), ''),
    (re.compile(r'&[a-zA-Z]+;'), ' '),
    (re.compile(r'\[\d+\]'), ''),
]

TABLE_PATTERNS = [
    (re.compile(r'^\s*\{\|.*$', re.MULTILINE), ''),
    (re.compile(r'^\s*\|\}.*$', re.MULTILINE), ''),
    (re.compile(r'^\s*\|-.*$', re.MULTILINE), ''),
    (re.compile(r'^\s*!.*$', re.MULTILINE), ''),
    (re.compile(r'^\|.*$', re.MULTILINE), ''),
]

SKIP_SECTIONS = {
    'सन्दर्भ', 'सन्दर्भहरू', 'बाह्य कडीहरू', 'बाह्य लिङ्कहरू',
    'श्रेणी', 'यो पनि हेर्नुहोस्', 'स्रोत', 'टिप्पणी',
    'references', 'external links', 'see also', 'notes',
}

def clean_wiki_article(text):
    for p, r in TABLE_PATTERNS:
        text = p.sub(r, text)
    for p, r in WIKI_MARKUP:
        text = p.sub(r, text)
    # Remove non-content sections
    lines = text.split('\n')
    result = []
    skip = False
    for line in lines:
        s = line.strip().lower()
        if s in SKIP_SECTIONS:
            skip = True
            continue
        if re.match(r'^={2,}.*={2,}$', s):
            skip = False
        if not skip:
            result.append(line)
    text = '\n'.join(result)
    # Normalize whitespace
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

print('✅ Wiki cleaning functions ready')

In [ ]:
# ── Process Wikipedia dump ──
MANUAL_DIR = f'{WORK_DIR}/manual'
os.makedirs(MANUAL_DIR, exist_ok=True)

NS = '{http://www.mediawiki.org/xml/export-0.10/}'

manual_docs = []
manual_total_chars = 0
manual_total_docs = 0
manual_shard_idx = 0
articles_seen = 0
skipped_in_auto = 0
rejected = 0

print('Processing Nepali Wikipedia dump...')
print('(Skipping articles already in automated corpus)\n')

with bz2.open(WIKI_DUMP, 'rt', encoding='utf-8') as f:
    context = ET.iterparse(f, events=('end',))
    for event, elem in context:
        if elem.tag == f'{NS}page':
            ns_elem = elem.find(f'{NS}ns')
            ns = ns_elem.text if ns_elem is not None else '0'
            if ns != '0':
                elem.clear()
                continue
            redirect = elem.find(f'{NS}redirect')
            if redirect is not None:
                elem.clear()
                continue
            
            title_elem = elem.find(f'{NS}title')
            id_elem = elem.find(f'{NS}id')
            revision = elem.find(f'{NS}revision')
            
            title = title_elem.text if title_elem is not None else ''
            page_id = id_elem.text if id_elem is not None else ''
            
            text_content = ''
            if revision is not None:
                text_elem = revision.find(f'{NS}text')
                if text_elem is not None and text_elem.text:
                    text_content = text_elem.text
            
            elem.clear()
            
            if not text_content.strip():
                continue
            
            articles_seen += 1
            
            # Clean
            cleaned = clean_wiki_article(text_content)
            if len(cleaned) < MIN_CHARS:
                rejected += 1
                continue
            if devanagari_ratio(cleaned) < MIN_DEV_RATIO:
                rejected += 1
                continue
            
            # Check if already in automated set
            h = text_hash(cleaned)
            if h in auto_hashes:
                skipped_in_auto += 1
                continue
            
            doc_id = hashlib.md5(f'{page_id}_{title}'.encode()).hexdigest()[:16]
            doc = {
                'doc_id': doc_id,
                'title': title,
                'wiki_page_id': page_id,
                'source': 'Nepali Wikipedia Dump',
                'source_type': 'wikipedia_dump',
                'collection_type': 'manual',
                'language': LANGUAGE,
                'text': cleaned,
                'clean_text_characters': len(cleaned),
                'devanagari_ratio': round(devanagari_ratio(cleaned), 4),
                'text_sha256': h,
            }
            
            manual_docs.append(doc)
            manual_total_chars += len(cleaned)
            manual_total_docs += 1
            
            if len(manual_docs) >= SHARD_SIZE:
                shard_file = f'{MANUAL_DIR}/wiki_shard_{manual_shard_idx:04d}.jsonl'
                with open(shard_file, 'w', encoding='utf-8') as sf:
                    for d in manual_docs:
                        sf.write(json.dumps(d, ensure_ascii=False) + '\n')
                print(f'  ✅ {shard_file}: {len(manual_docs):,} docs')
                manual_docs = []
                manual_shard_idx += 1
            
            if articles_seen % 5000 == 0:
                print(f'  Seen: {articles_seen:,} | Accepted: {manual_total_docs:,} | '
                      f'Already in auto: {skipped_in_auto:,} | Rejected: {rejected:,}')

# Flush remaining
if manual_docs:
    shard_file = f'{MANUAL_DIR}/wiki_shard_{manual_shard_idx:04d}.jsonl'
    with open(shard_file, 'w', encoding='utf-8') as sf:
        for d in manual_docs:
            sf.write(json.dumps(d, ensure_ascii=False) + '\n')
    print(f'  ✅ {shard_file}: {len(manual_docs):,} docs')

print(f'\n{"="*60}')
print(f'WIKI DUMP PROCESSING COMPLETE')
print(f'  Articles seen:          {articles_seen:,}')
print(f'  Accepted (manual):      {manual_total_docs:,}')
print(f'  Already in automated:   {skipped_in_auto:,}')
print(f'  Rejected (quality):     {rejected:,}')
print(f'  Manual chars:           {manual_total_chars:,}')
print(f'  Est. manual tokens:     ~{manual_total_chars // 6:,}')
print(f'{"="*60}')

In [ ]:
# Also include our existing 5,000 manually scraped docs if available
# (These were collected via Wikimedia API earlier)
EXISTING_MANUAL = '/content/drive/MyDrive/24KaratAu/nepali/corpus/manual'
existing_manual_count = 0
existing_manual_chars = 0

if os.path.exists(EXISTING_MANUAL):
    for sf in sorted(Path(EXISTING_MANUAL).glob('shard-*.jsonl')):
        with open(sf, 'r', encoding='utf-8') as f:
            for line in f:
                if not line.strip(): continue
                doc = json.loads(line)
                text = doc.get('text', '') or doc.get('clean_text', '')
                if not text:
                    # Try reading from text path
                    tp = doc.get('clean_text_path') or doc.get('text_path')
                    if tp and os.path.exists(tp):
                        text = open(tp, 'r', encoding='utf-8').read()
                if not text or len(text) < MIN_CHARS:
                    continue
                h = text_hash(text)
                if h in auto_hashes:
                    continue  # Already in automated
                doc['collection_type'] = 'manual'
                doc['text'] = text
                doc['text_sha256'] = h
                doc['clean_text_characters'] = len(text)
                # Write to manual dir
                existing_manual_count += 1
                existing_manual_chars += len(text)
    print(f'Found {existing_manual_count:,} existing manual docs ({existing_manual_chars:,} chars)')
else:
    print('No existing manual shards found on Drive (that is OK)')

---
## Step 3: Merge + Deduplicate

In [ ]:
# Merge all docs: manual first (priority), then automated
MERGED_DIR = f'{WORK_DIR}/merged'
os.makedirs(MERGED_DIR, exist_ok=True)

all_hashes = set()
all_docs_manual = []
all_docs_auto = []

# Load manual shards
for sf in sorted(Path(f'{WORK_DIR}/manual').glob('*.jsonl')):
    with open(sf, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            doc = json.loads(line)
            h = doc.get('text_sha256', '')
            if h and h not in all_hashes:
                all_hashes.add(h)
                all_docs_manual.append(doc)

# Load automated shards
for sf in sorted(Path(f'{WORK_DIR}/automated').glob('*.jsonl')):
    with open(sf, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            doc = json.loads(line)
            h = doc.get('text_sha256', '')
            if h and h not in all_hashes:
                all_hashes.add(h)
                all_docs_auto.append(doc)

all_docs = all_docs_manual + all_docs_auto
manual_chars = sum(d['clean_text_characters'] for d in all_docs_manual)
auto_chars = sum(d['clean_text_characters'] for d in all_docs_auto)
total_chars = manual_chars + auto_chars

print(f'Merged corpus:')
print(f'  Manual:    {len(all_docs_manual):,} docs ({manual_chars:,} chars)')
print(f'  Automated: {len(all_docs_auto):,} docs ({auto_chars:,} chars)')
print(f'  Total:     {len(all_docs):,} docs ({total_chars:,} chars)')
if total_chars > 0:
    print(f'  Manual %:  {manual_chars/total_chars*100:.1f}%')

# Write merged shards
for i in range(0, len(all_docs), SHARD_SIZE):
    batch = all_docs[i:i+SHARD_SIZE]
    shard_file = f'{MERGED_DIR}/merged_shard_{i//SHARD_SIZE:04d}.jsonl'
    with open(shard_file, 'w', encoding='utf-8') as f:
        for d in batch:
            f.write(json.dumps(d, ensure_ascii=False) + '\n')

print(f'\n✅ Merged shards written to {MERGED_DIR}')

---
## Step 4: Train/Val/Test Split

In [ ]:
# Stratified split by source_type: 90% train / 5% val / 5% test
SPLITS_DIR = f'{WORK_DIR}/splits'
os.makedirs(SPLITS_DIR, exist_ok=True)

random.seed(42)

# Group by source
source_groups = {}
for doc in all_docs:
    s = doc.get('source_type', 'unknown')
    source_groups.setdefault(s, []).append(doc)

train, val, test = [], [], []
for source, docs in source_groups.items():
    random.shuffle(docs)
    n = len(docs)
    n_val = max(1, int(n * 0.05)) if n > 2 else 0
    n_test = max(1, int(n * 0.05)) if n > 2 else 0
    n_train = n - n_val - n_test
    train.extend(docs[:n_train])
    val.extend(docs[n_train:n_train+n_val])
    test.extend(docs[n_train+n_val:])

random.shuffle(train)
random.shuffle(val)
random.shuffle(test)

# Write JSONL splits
for name, docs in [('train', train), ('val', val), ('test', test)]:
    with open(f'{SPLITS_DIR}/{name}.jsonl', 'w', encoding='utf-8') as f:
        for d in docs:
            f.write(json.dumps(d, ensure_ascii=False) + '\n')

# Write plain text for tokenizer training
for name, docs in [('train', train), ('val', val), ('test', test)]:
    with open(f'{SPLITS_DIR}/{name}.txt', 'w', encoding='utf-8') as f:
        for d in docs:
            text = d.get('text', '')
            if text.strip():
                f.write(text.strip() + '\n\n')

for name, docs in [('train', train), ('val', val), ('test', test)]:
    chars = sum(d['clean_text_characters'] for d in docs)
    manual_n = sum(1 for d in docs if d.get('collection_type') == 'manual')
    print(f'  {name:6s}: {len(docs):>8,} docs | {chars:>14,} chars | manual: {manual_n:,}')

print(f'\n✅ Splits written to {SPLITS_DIR}')

---
## Step 5: Train BPE Tokenizer

In [ ]:
import sentencepiece as spm

TOKENIZER_DIR = f'{WORK_DIR}/tokenizer'
os.makedirs(TOKENIZER_DIR, exist_ok=True)

MODEL_PREFIX = f'{TOKENIZER_DIR}/nepali_bpe_32k'
TRAIN_TXT = f'{SPLITS_DIR}/train.txt'

train_size_mb = os.path.getsize(TRAIN_TXT) / (1024*1024)
print(f'Training BPE tokenizer on {TRAIN_TXT} ({train_size_mb:.1f} MB)...')
print('This may take 5-15 minutes...\n')

spm.SentencePieceTrainer.train(
    input=TRAIN_TXT,
    model_prefix=MODEL_PREFIX,
    vocab_size=32000,
    model_type='bpe',
    character_coverage=0.9995,
    num_threads=4,
    max_sentence_length=16384,
    input_sentence_size=10_000_000,
    shuffle_input_sentence=True,
    pad_id=0, unk_id=1, bos_id=2, eos_id=3,
    pad_piece='<pad>', unk_piece='<unk>',
    bos_piece='<bos>', eos_piece='<eos>',
    split_by_unicode_script=True,
    split_by_whitespace=True,
    split_digits=True,
    byte_fallback=True,
)

print(f'\n✅ Tokenizer trained!')
print(f'  Model: {MODEL_PREFIX}.model')
print(f'  Vocab: {MODEL_PREFIX}.vocab')

# Quick test
sp = spm.SentencePieceProcessor()
sp.load(f'{MODEL_PREFIX}.model')
print(f'  Vocab size: {sp.get_piece_size():,}')

test_text = 'नेपाल एक दक्षिण एशियाली राष्ट्र हो।'
tokens = sp.encode(test_text, out_type=str)
print(f'\n  Test: {test_text}')
print(f'  Tokens ({len(tokens)}): {tokens}')

---
## Step 6: Tokenize Corpus + Count Tokens

In [ ]:
# Count exact tokens for each split
sp = spm.SentencePieceProcessor()
sp.load(f'{MODEL_PREFIX}.model')

token_counts = {}
for split_name in ['train', 'val', 'test']:
    txt_file = f'{SPLITS_DIR}/{split_name}.txt'
    total_tokens = 0
    total_docs = 0
    
    with open(txt_file, 'r', encoding='utf-8') as f:
        current_doc = []
        for line in f:
            if line.strip():
                current_doc.append(line.strip())
            else:
                if current_doc:
                    text = ' '.join(current_doc)
                    ids = sp.encode(text, out_type=int)
                    total_tokens += len(ids)
                    total_docs += 1
                    current_doc = []
                    if total_docs % 50000 == 0:
                        print(f'  {split_name}: {total_docs:,} docs, {total_tokens:,} tokens')
        if current_doc:
            text = ' '.join(current_doc)
            ids = sp.encode(text, out_type=int)
            total_tokens += len(ids)
            total_docs += 1
    
    token_counts[split_name] = {'docs': total_docs, 'tokens': total_tokens}
    print(f'  {split_name}: {total_docs:,} docs → {total_tokens:,} tokens')

total_all = sum(v['tokens'] for v in token_counts.values())
print(f'\n{"="*60}')
print(f'TOTAL NEPALI TOKENS: {total_all:,}')
print(f'{"="*60}')

---
## Step 7: Upload Everything to Google Drive

In [ ]:
# Copy all outputs to Google Drive
import shutil

# Cleaned shards (manual + automated + merged)
for subdir in ['manual', 'automated', 'merged']:
    src = f'{WORK_DIR}/{subdir}'
    dst = f'{DRIVE_BASE}/cleaned/{subdir}'
    if os.path.exists(src):
        os.makedirs(dst, exist_ok=True)
        for f in Path(src).glob('*.jsonl'):
            shutil.copy2(str(f), dst)
        print(f'  ✅ Copied {subdir}/ to Drive')

# Splits
src = f'{WORK_DIR}/splits'
dst = f'{DRIVE_BASE}/splits'
if os.path.exists(src):
    for f in Path(src).iterdir():
        shutil.copy2(str(f), dst)
    print(f'  ✅ Copied splits/ to Drive')

# Tokenizer
src = f'{WORK_DIR}/tokenizer'
dst = f'{DRIVE_BASE}/tokenizer'
if os.path.exists(src):
    for f in Path(src).iterdir():
        shutil.copy2(str(f), dst)
    print(f'  ✅ Copied tokenizer/ to Drive')

# Save final summary
summary = {
    'language': LANGUAGE,
    'manual_docs': len(all_docs_manual),
    'manual_chars': manual_chars,
    'automated_docs': len(all_docs_auto),
    'automated_chars': auto_chars,
    'total_docs': len(all_docs),
    'total_chars': total_chars,
    'manual_percentage': round(manual_chars / max(1, total_chars) * 100, 2),
    'token_counts': token_counts,
    'total_tokens': total_all,
    'tokenizer_vocab_size': sp.get_piece_size(),
}

with open(f'{DRIVE_BASE}/corpus_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f'\n{"="*60}')
print(f'🎉 NEPALI CORPUS COMPLETE!')
print(f'  Total tokens:  {total_all:,}')
print(f'  Manual %:      {manual_chars/max(1,total_chars)*100:.1f}%')
print(f'  Drive path:    {DRIVE_BASE}')
print(f'{"="*60}')